# Gayaku garment classifier — MobileNetV3-small on the Fashion Product Images catalogue

Trains the on-device classifier for the Smart Scanner (docs/adr/0001, 0003, 0005) and
exports the artifacts the Flutter app needs:

| artifact | contract |
|---|---|
| `artifacts/garment_classifier.tflite` | **the shipped model.** Input `image` **NHWC `[1,224,224,3]` float32 RGB in `[0,1]`**; output `scores` `[1,N]` softmax whose class order is exactly `classes.json`'s `labels`, which is what `flutter_app/lib/logic/garment_labels.dart` mirrors |
| `artifacts/garment_classifier.onnx` | audit copy of the same graph (same IO contract), for checking the export outside Flutter |
| `artifacts/classes.json` | class list: `labels[i]` is the label score `i` means |
| `artifacts/report.md` | metrics: per-class accuracy and the confusion matrix on the held-out 10% |

The ImageNet normalisation is **baked into the exported graph** by a wrapper module, so the
Dart side feeds raw `[0,1]` pixels and the training-time and inference-time preprocessing
cannot drift apart.

### Before you run
1. **Add Input** → `paramaggarwal/fashion-product-images-small` (same `styles.csv` as the full
   variant, 60k images: the full 25 GB set trains no better at 224 px).
2. **Add Input** → a dataset holding `ml_workspace/labels.json` (upload the repo file as a
   private dataset). It is the single source of the class list and the Indonesian names.
   Optional: also attach `ml_workspace/data/{train,val,test}.csv` to reuse the split that
   `prepare_data.py` computed locally instead of recomputing it here.
3. Settings: GPU on, Internet on (only for the one `pip install litert-torch`).
4. Run all, then download the four artifacts and follow `ml_workspace/README.md`.

Coursework only: the catalogue's MIT tag covers its *labels*, not the photographs, so these
weights stay in this project — see `flutter_app/assets/models/LICENSES.md`.


In [ ]:
"""Config, paths, reproducibility."""
import csv
import json
import random
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

SEED = 20261015
DATASET_ID = "paramaggarwal/fashion-product-images-dataset"
KAGGLE_INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
ARTIFACTS = WORK / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 128
EPOCHS = 12
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
VAL_FRAC, TEST_FRAC = 0.10, 0.10
MIN_IMAGES_PER_CLASS = 8  # same rule as ml_workspace/scripts/prepare_data.py
IMAGE_MEAN = (0.485, 0.456, 0.406)  # ImageNet; baked into the exported model
IMAGE_STD = (0.229, 0.224, 0.225)
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("device", DEVICE, "| torch", torch.__version__, "| cuda", torch.version.cuda)


In [ ]:
def find_one(name):
    """First file called `name` under the attached inputs, then the working dir."""
    for root in (KAGGLE_INPUT, Path.cwd(), WORK):
        if root.exists():
            hits = sorted(p for p in root.rglob(name) if p.is_file())
            if hits:
                return hits[0]
    raise FileNotFoundError(
        f"{name} not found: attach the dataset that holds ml_workspace/labels.json and "
        "paramaggarwal/fashion-product-images-small as notebook inputs"
    )


labels_path = find_one("labels.json")
styles_path = find_one("styles.csv")
catalogue_root = styles_path.parent
images_root = catalogue_root / "images"
assert images_root.is_dir(), f"no images/ next to {styles_path}"

doc = json.loads(labels_path.read_text(encoding="utf-8"))
entries = [e for e in doc["labels"] if e.get("dataset") == DATASET_ID and e.get("keep")]
classes = [e["label"] for e in entries]
assert classes, f"labels.json has no keep:true entries for {DATASET_ID}"
names = {e["label"]: e["name"] for e in entries}
categories = {e["label"]: e["category"] for e in entries}
print(f"labels.json {labels_path} -> {len(classes)} classes, {len(doc['labels'])} catalogue labels")
print(f"catalogue {catalogue_root}")
print("first classes", classes[:8])


In [ ]:
def stratified_split(rows, seed, val_frac, test_frac):
    """Deterministic 80/10/10 by articleType; mirrors ml_workspace/scripts/prepare_data.py."""
    rng = random.Random(seed)
    splits = {"train": [], "val": [], "test": []}
    for label in sorted({row[2] for row in rows}):
        group = [row for row in rows if row[2] == label]
        rng.shuffle(group)
        n_test = max(1, round(len(group) * test_frac))
        n_val = max(1, round(len(group) * val_frac))
        assert len(group) - n_test - n_val >= 1, f"{label}: {len(group)} photographs"
        splits["test"] += group[:n_test]
        splits["val"] += group[n_test:n_test + n_val]
        splits["train"] += group[n_test + n_val:]
    return {name: sorted(rows, key=lambda row: (row[2], row[0])) for name, rows in splits.items()}


def load_manifest(name):
    try:
        path = find_one(f"{name}.csv")
    except FileNotFoundError:
        return None
    with path.open(newline="", encoding="utf-8") as handle:
        return [(r["id"], r["image"], r["label"]) for r in csv.DictReader(handle)]


splits = {name: load_manifest(name) for name in ("train", "val", "test")}
if all(splits.values()):
    print("reusing the manifests prepare_data.py wrote")
else:
    kept = set(classes)
    rows = []
    with styles_path.open(encoding="utf-8", errors="replace", newline="") as handle:
        for record in csv.DictReader(handle):
            article_id = (record.get("id") or "").strip()
            label = (record.get("articleType") or "").strip()
            if article_id and label in kept and (images_root / f"{article_id}.jpg").is_file():
                rows.append((article_id, f"images/{article_id}.jpg", label))
    counts = Counter(row[2] for row in rows)
    thin = sorted(
        (label, counts.get(label, 0))
        for label in classes
        if counts.get(label, 0) < MIN_IMAGES_PER_CLASS
    )
    assert not thin, f"labels.json keeps these, but they have too few photographs: {thin}"
    splits = stratified_split(rows, SEED, VAL_FRAC, TEST_FRAC)
    print("computed the split from styles.csv")

every_row = splits["train"] + splits["val"] + splits["test"]
classes_doc = {
    "version": 1,
    "dataset": DATASET_ID,
    "source": f"{DATASET_ID} styles.csv filtered by labels.json keep:true, seed {SEED}",
    "labels": classes,
    "counts": {label: sum(1 for row in every_row if row[2] == label) for label in classes},
    "splits": {name: len(rows) for name, rows in splits.items()},
}
(ARTIFACTS / "classes.json").write_text(
    json.dumps(classes_doc, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print({name: len(rows) for name, rows in splits.items()})


In [ ]:
class Catalogue(Dataset):
    """One row per photograph; `image` is the path relative to the catalogue root."""

    def __init__(self, rows, train):
        self.rows = rows
        self.index = {label: i for i, label in enumerate(classes)}
        resize = [transforms.Resize(256)]
        if train:
            jitter = [
                transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.2, 0.2, 0.2),
            ]
        else:
            jitter = [transforms.CenterCrop(IMAGE_SIZE)]
        self.transform = transforms.Compose(
            resize
            + jitter
            + [
                transforms.ToTensor(),
                transforms.Normalize(IMAGE_MEAN, IMAGE_STD),
            ]
            + ([transforms.RandomErasing(p=0.25, value=0)] if train else [])
        )

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        _, relative, label = self.rows[index]
        image = Image.open(catalogue_root / relative).convert("RGB")
        return self.transform(image), self.index[label]


loaders = {
    name: DataLoader(
        Catalogue(rows, train=name == "train"),
        batch_size=BATCH_SIZE,
        shuffle=name == "train",
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        drop_last=name == "train",
    )
    for name, rows in splits.items()
}
probe_images, probe_labels = next(iter(loaders["train"]))
print("batch", tuple(probe_images.shape), "labels", tuple(probe_labels.shape), "classes", len(classes))


In [ ]:
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(classes))
model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except TypeError:  # torch < 2.4
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters, {len(classes)} classes")


In [ ]:
def run_epoch(loader, train):
    model.train(train)
    total = correct = 0
    loss_sum = 0.0
    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train), torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, targets)
        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        loss_sum += loss.detach().item() * targets.size(0)
        correct += (logits.detach().argmax(1) == targets).sum().item()
        total += targets.size(0)
    return loss_sum / max(total, 1), correct / max(total, 1)


history = []
best_val = 0.0
checkpoint = ARTIFACTS / "garment_classifier.pt"
for epoch in range(1, EPOCHS + 1):
    started = time.time()
    train_loss, train_acc = run_epoch(loaders["train"], train=True)
    val_loss, val_acc = run_epoch(loaders["val"], train=False)
    schedule.step()
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )
    if val_acc > best_val:
        best_val = val_acc
        torch.save({"model": model.state_dict(), "classes": classes, "epoch": epoch}, checkpoint)
    print(
        f"epoch {epoch:2d}/{EPOCHS}  train {train_loss:.3f}/{train_acc:.3f}  "
        f"val {val_loss:.3f}/{val_acc:.3f}  ({time.time() - started:.0f}s)"
    )
print(f"best val accuracy {best_val:.4f} -> {checkpoint}")


In [ ]:
state = torch.load(checkpoint, map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model"])


def predict(loader):
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for images, targets in loader:
            preds.append(model(images.to(DEVICE)).argmax(1).cpu())
            truths.append(targets)
    return torch.cat(preds).numpy(), torch.cat(truths).numpy()


def confusion_of(preds, truths):
    matrix = np.zeros((len(classes), len(classes)), dtype=np.int64)
    for truth, pred in zip(truths, preds):
        matrix[truth, pred] += 1
    return matrix


val_preds, val_truths = predict(loaders["val"])
test_preds, test_truths = predict(loaders["test"])
confusion = confusion_of(test_preds, test_truths)
val_accuracy = float((val_preds == val_truths).mean())
test_accuracy = float((test_preds == test_truths).mean())

per_class = []
for index, label in enumerate(classes):
    support = int(confusion[index].sum())
    accuracy = float(confusion[index, index] / support) if support else float("nan")
    per_class.append((label, names.get(label, label), categories.get(label, ""), support, accuracy))

print(f"val accuracy  {val_accuracy:.4f}  ({len(val_truths)} photographs)")
print(f"test accuracy {test_accuracy:.4f}  ({len(test_truths)} photographs)\n")
print(f"{'label':<22}{'name':<22}{'category':<12}{'n':>6}{'acc':>8}")
for label, name, category, support, accuracy in per_class:
    print(f"{label:<22}{name:<22}{category:<12}{support:>6}{accuracy:>8.3f}")

print("\nper-class accuracy, worst first")
for label, name, _, support, accuracy in sorted(per_class, key=lambda row: (row[4], -row[3])):
    if support:
        print(f"  {accuracy:.3f}  {label} ({name}, n={support})")

print("\nconfusion matrix: rows = truth, columns = prediction, order = classes.json")
width = max(len(label) for label in classes)
print(" " * width + "".join(f"{i:>5}" for i in range(len(classes))))
for index, label in enumerate(classes):
    print(f"{label:<{width}}" + "".join(f"{value:>5}" for value in confusion[index]))

confusion_png = ARTIFACTS / "confusion_matrix.png"
try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(figsize=(16, 16))
    image = axes.imshow(confusion, cmap="viridis")
    axes.set_xlabel("prediction")
    axes.set_ylabel("truth")
    axes.set_xticks(range(len(classes)))
    axes.set_yticks(range(len(classes)))
    axes.set_xticklabels(classes, rotation=90, fontsize=5)
    axes.set_yticklabels(classes, fontsize=5)
    figure.colorbar(image, fraction=0.04)
    figure.tight_layout()
    figure.savefig(confusion_png, dpi=120)
    print(f"\nwrote {confusion_png}")
except ImportError:
    print("\nmatplotlib missing; the matrix is above as text")

lines = [
    "# Garment classifier training report",
    "",
    f"* dataset: `{DATASET_ID}`, `styles.csv` filtered by `labels.json` (`keep: true`)",
    f"* classes: {len(classes)}",
    f"* photographs: train {len(splits['train'])}, val {len(splits['val'])}, test {len(splits['test'])}",
    f"* split: stratified 80/10/10 by `articleType`, seed {SEED} (same rule as `prepare_data.py`)",
    f"* model: torchvision MobileNetV3-small, ImageNet weights, new {len(classes)}-way head",
    f"* input: NHWC `[1,{IMAGE_SIZE},{IMAGE_SIZE},3]` float32 RGB in `[0,1]`; "
    "ImageNet normalisation baked into the exported graph",
    f"* training: {EPOCHS} epochs, batch {BATCH_SIZE}, AdamW lr {LEARNING_RATE}, "
    f"cosine schedule, label smoothing {LABEL_SMOOTHING}, augmented (crop, flip, colour jitter, erasing)",
    f"* accuracy: val {val_accuracy:.4f}, test {test_accuracy:.4f}",
    "",
    "## Per-class accuracy (test split)",
    "",
    "| label | name | category | n | accuracy |",
    "|---|---|---|---:|---:|",
]
lines += [
    f"| {label} | {name} | {category} | {support} | {accuracy:.3f} |"
    for label, name, category, support, accuracy in per_class
]
lines += [
    "",
    "## Confusion matrix (test split)",
    "",
    "Rows are the truth, columns the prediction, order is `classes.json`:",
    "",
    "```",
    " " * width + "".join(f"{i:>5}" for i in range(len(classes))),
]
lines += [
    f"{label:<{width}}" + "".join(f"{value:>5}" for value in confusion[index])
    for index, label in enumerate(classes)
]
lines += [
    "```",
    "",
    f"![confusion matrix]({confusion_png.name})" if confusion_png.exists() else "",
    "",
    "## Epochs",
    "",
    "| epoch | train loss | train acc | val loss | val acc |",
    "|---:|---:|---:|---:|---:|",
]
lines += [
    f"| {row['epoch']} | {row['train_loss']:.4f} | {row['train_acc']:.4f} | "
    f"{row['val_loss']:.4f} | {row['val_acc']:.4f} |"
    for row in history
]
lines += [
    "",
    "Weights trained by us. The catalogue's MIT tag covers its labels, not the photographs, "
    "so the artifacts are coursework-only and are not redistributed.",
    "",
]
report_path = ARTIFACTS / "report.md"
report_path.write_text("\n".join(line for line in lines if line is not None) + "\n", encoding="utf-8")
print(f"wrote {report_path}")


In [ ]:
class ExportClassifier(nn.Module):
    """NHWC `[1,224,224,3]` RGB in `[0,1]` -> `[1,N]` softmax.

    The training normalisation lives inside, so the Dart side only builds `[0,1]` pixels
    and the two preprocessing paths cannot drift apart.
    """

    def __init__(self, backbone, mean=IMAGE_MEAN, std=IMAGE_STD):
        super().__init__()
        self.backbone = backbone.eval()
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, image):
        pixels = image.permute(0, 3, 1, 2)  # NHWC -> NCHW
        return torch.softmax(self.backbone((pixels - self.mean) / self.std), dim=1)


wrapper = ExportClassifier(model).to(DEVICE).eval()
sample = torch.rand(1, IMAGE_SIZE, IMAGE_SIZE, 3, device=DEVICE)
onnx_path = ARTIFACTS / "garment_classifier.onnx"
export_kwargs = dict(input_names=["image"], output_names=["scores"], opset_version=17)
try:
    torch.onnx.export(wrapper, (sample,), onnx_path, dynamo=False, **export_kwargs)
except TypeError:  # older torch has no `dynamo` switch
    torch.onnx.export(wrapper, (sample,), onnx_path, **export_kwargs)
print(f"{onnx_path} ({onnx_path.stat().st_size / 1e6:.2f} MB)")

import onnxruntime as ort

session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
probe = np.random.rand(1, IMAGE_SIZE, IMAGE_SIZE, 3).astype(np.float32)
onnx_out = session.run(["scores"], {"image": probe})[0]
with torch.no_grad():
    torch_out = wrapper(torch.from_numpy(probe).to(DEVICE)).cpu().numpy()
print("inputs ", [(item.name, item.shape, item.type) for item in session.get_inputs()])
print("outputs", [(item.name, item.shape, item.type) for item in session.get_outputs()])
print(f"max |onnx - torch| = {np.abs(onnx_out - torch_out).max():.2e}")


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "litert-torch"], check=True)
try:
    import litert_torch as converter  # the package was renamed from ai-edge-torch
except ImportError:
    import ai_edge_torch as converter

tflite_path = ARTIFACTS / "garment_classifier.tflite"
converter.convert(wrapper, (sample,)).export(str(tflite_path))
print(f"{tflite_path} ({tflite_path.stat().st_size / 1e6:.2f} MB)")


def read_tflite(path):
    """Minimal TFLite flatbuffer reader over Model/SubGraph/Tensor (see scripts/tflite_io.py)."""
    blob = Path(path).read_bytes()
    assert blob[4:8] == b"TFL3", "not a TFLite flatbuffer: magic bytes must be TFL3 at offset 4"

    def u16(pos):
        return int.from_bytes(blob[pos:pos + 2], "little")

    def u32(pos):
        return int.from_bytes(blob[pos:pos + 4], "little")

    def i32(pos):
        return int.from_bytes(blob[pos:pos + 4], "little", signed=True)

    class Table:
        def __init__(self, pos):
            self.pos = pos

        def field(self, index):
            vtable = self.pos - i32(self.pos)
            if 4 + 2 * index + 2 > u16(vtable):
                return None
            relative = u16(vtable + 4 + 2 * index)
            return None if relative == 0 else self.pos + relative

        def indirect(self, pos):
            return pos + u32(pos)

        def string(self, index):
            pos = self.field(index)
            if pos is None:
                return None
            start = self.indirect(pos)
            return blob[start + 4:start + 4 + u32(start)].decode()

        def integers(self, index):
            pos = self.field(index)
            if pos is None:
                return []
            start = self.indirect(pos)
            return [i32(start + 4 + 4 * i) for i in range(u32(start))]

        def tables(self, index):
            pos = self.field(index)
            if pos is None:
                return []
            start = self.indirect(pos)
            return [Table(self.indirect(start + 4 + 4 * i)) for i in range(u32(start))]

    dtypes = {0: "float32", 1: "float16", 2: "int32", 3: "uint8", 9: "int8"}
    subgraph = Table(u32(0)).tables(2)[0]
    tensors = subgraph.tables(0)

    def describe(index):
        tensor = tensors[index]
        raw = tensor.field(1)
        return {
            "name": tensor.string(3),
            "shape": tensor.integers(0),
            "dtype": dtypes.get(0 if raw is None else blob[raw], "unknown"),
        }

    return {
        "inputs": [describe(i) for i in subgraph.integers(1)],
        "outputs": [describe(i) for i in subgraph.integers(2)],
    }


record = read_tflite(tflite_path)
for kind in ("inputs", "outputs"):
    for index, tensor in enumerate(record[kind]):
        print(f"{kind[:-1]} {index}: name={tensor['name']!r} shape={tensor['shape']} dtype={tensor['dtype']}")
assert record["inputs"][0]["shape"] == [1, IMAGE_SIZE, IMAGE_SIZE, 3], record["inputs"][0]
assert record["inputs"][0]["dtype"] == "float32", record["inputs"][0]
assert record["outputs"][0]["shape"] == [1, len(classes)], record["outputs"][0]
assert record["outputs"][0]["dtype"] == "float32", record["outputs"][0]
print("TFLite IO matches the frozen contract in docs/adr/0005 + ml_workspace/README.md")

try:
    import tensorflow as tf

    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    interpreter.set_tensor(interpreter.get_input_details()[0]["index"], probe)
    interpreter.invoke()
    tflite_out = interpreter.get_tensor(interpreter.get_output_details()[0]["index"])
    print(f"max |tflite - onnx| = {np.abs(tflite_out - onnx_out).max():.2e}")
except ImportError:
    print("tensorflow is not importable here; skipped the numeric tflite <-> onnx comparison")


In [ ]:
print("artifacts to download from /kaggle/working/artifacts:")
for path in sorted(ARTIFACTS.iterdir()):
    print(f"  {path.name:<32}{path.stat().st_size / 1e6:8.2f} MB")
print()
print("copy garment_classifier.tflite to flutter_app/assets/models/garment_classifier.tflite")
print("copy classes.json to ml_workspace/artifacts/classes.json")
print("then: python ml_workspace/scripts/gen_dart_labels.py")


## After the run

1. Download `artifacts/` and put the files where the app expects them:
   * `garment_classifier.tflite` → `flutter_app/assets/models/garment_classifier.tflite`
     (the asset directory is already listed in `pubspec.yaml`)
   * `classes.json` → `ml_workspace/artifacts/classes.json`
   * `report.md` → `ml_workspace/artifacts/report.md`
2. **Regenerate the Dart label file** — the codegen rule is that
   `garmentCategories[i]` must name the class that output `scores[i]` means:

   ```bash
   python ml_workspace/scripts/gen_dart_labels.py            # rewrites flutter_app/lib/logic/garment_labels.dart
   python ml_workspace/scripts/gen_dart_labels.py --check    # fails if it is stale
   python ml_workspace/scripts/tflite_io.py flutter_app/assets/models/garment_classifier.tflite
   ```

   `gen_dart_labels.py` exits non-zero if a class in `classes.json` has no entry in
   `ml_workspace/labels.json`, so a new class can never reach the app without a Category
   and an Indonesian name.
3. Commit the model, `classes.json` and the regenerated Dart file together, so the class
   order and the shipped weights can never disagree.
